# 5m

In [1]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from tqdm.auto import tqdm


# =========================================================
# CONFIG
# =========================================================
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
OUT_DIR = POLY_DATA_DIR / "polymarket_meta_5m"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = {
    "btc": "bitcoin",
    "eth": "ethereum",
    "sol": "solana",
    "xrp": "xrp",
}

START_DATETIME_UTC = "2026-02-19 00:00:00"   # ← 원하는 시작일로 변경
END_DATETIME_UTC   = "2026-05-01 00:00:00"                   

GAMMA_EVENT_BY_SLUG  = "https://gamma-api.polymarket.com/events/slug"
CLOB_PRICE_HISTORY   = "https://clob.polymarket.com/prices-history"

TIMEOUT             = 12
MAX_WORKERS         = 32
SLEEP_BETWEEN_CALLS = 0.00
FIDELITY            = 1
INTERVAL_SEC        = 5 * 60   # 300초

PKL_PATH     = OUT_DIR / "5m_closed_only.pkl"
MISSING_PATH = OUT_DIR / "5m_missing.pkl"
SUMMARY_PATH = OUT_DIR / "5m_closed_only_summary.csv"


# =========================================================
# TIME UTILS
# =========================================================
def parse_utc(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)

def dt_to_epoch(dt: datetime) -> int:
    return int(dt.timestamp())

def floor_to_5m_epoch(ts: int) -> int:
    return ts - (ts % INTERVAL_SEC)

def epoch_to_slot_utc(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()

def generate_slug(asset_short: str, slot_epoch: int) -> str:
    # 5분봉: {asset_short}-updown-5m-{slot_epoch}
    return f"{asset_short}-updown-5m-{slot_epoch}"

def iso_to_epoch(ts: str) -> int:
    return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp())

def minute_elapsed_dict(start_ts: int, history: list, window_minutes: int = 5):
    """5분봉: startTs 기준 경과 분 (0~5)"""
    d = {}
    for h in history:
        m_elapsed = int((h["t"] - start_ts) // 60)
        if 0 <= m_elapsed <= window_minutes:
            d[m_elapsed] = h["p"]
    return d


# =========================================================
# 기존 데이터 로드
# =========================================================
def load_existing() -> tuple[pd.DataFrame, set]:
    if PKL_PATH.exists():
        try:
            df = pd.read_pickle(PKL_PATH)
            existing = set(zip(df["asset"], df["slot_epoch"]))
            print(f"[Load] 기존 데이터 {len(df)}행 로드 완료 ({len(existing)}개 (asset, slot) 쌍)")
            return df, existing
        except Exception as e:
            print(f"[Load] pkl 로드 실패: {e} → 빈 상태로 시작")
    return pd.DataFrame(), set()


# =========================================================
# HTTP
# =========================================================
def fetch_event_meta(slug: str):
    try:
        r = requests.get(f"{GAMMA_EVENT_BY_SLUG}/{slug}", timeout=TIMEOUT)
        if r.status_code != 200:
            return None, r.status_code
        return r.json(), 200
    except Exception:
        return None, -1

def extract_market_info(event_data: dict) -> dict | None:
    markets = event_data.get("markets")
    if not markets:
        return None
    m     = markets[0]
    clobs = m.get("clobTokenIds")
    if clobs is None:
        return None
    try:
        vol_num = float(m.get("volume")) if m.get("volume") is not None else None
    except Exception:
        vol_num = None
    return {
        "clobTokenIds":  clobs,
        "outcomes":      m.get("outcomes"),
        "outcomePrices": m.get("outcomePrices"),
        "volume":        vol_num,
    }

def fetch_price_history(token_id: str, start_ts: int, end_ts: int):
    params = {"market": token_id, "startTs": start_ts, "endTs": end_ts, "fidelity": FIDELITY}
    try:
        r = requests.get(CLOB_PRICE_HISTORY, params=params, timeout=TIMEOUT)
        if r.status_code != 200:
            return None
        return r.json().get("history", [])
    except Exception:
        return None


# =========================================================
# STEP 1: Meta 수집
# =========================================================
def collect_meta_one_slot(asset_short: str, asset_full: str, slot_epoch: int):
    slug     = generate_slug(asset_short, slot_epoch)
    slot_utc = epoch_to_slot_utc(slot_epoch)

    if SLEEP_BETWEEN_CALLS:
        time.sleep(SLEEP_BETWEEN_CALLS)

    event_data, status = fetch_event_meta(slug)

    if event_data is None:
        return False, {
            "asset": asset_full, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": status, "reason": "http_failed",
        }

    market_info = extract_market_info(event_data)
    if market_info is None:
        return False, {
            "asset": asset_full, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": 200, "reason": "no_markets_in_event",
        }

    return True, {
        "asset": asset_full, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
        "slug": slug, **market_info,
    }


# =========================================================
# STEP 2: Price History 수집
# =========================================================
def collect_price_history_one_row(row: dict):
    try:
        clobs          = json.loads(row["clobTokenIds"]) if isinstance(row["clobTokenIds"], str) else row["clobTokenIds"]
        outcomes       = json.loads(row["outcomes"])     if isinstance(row["outcomes"], str)     else row["outcomes"]
        outcome_prices = json.loads(row["outcomePrices"])if isinstance(row["outcomePrices"], str)else row["outcomePrices"]

        yes_idx   = outcomes.index("Up") if "Up" in outcomes else 0
        no_idx    = 1 - yes_idx
        token_yes = clobs[yes_idx]
        token_no  = clobs[no_idx]

        # 5분봉: startTs = slot_epoch, endTs = slot_epoch + 300
        start_ts = int(row["slot_epoch"])
        end_ts   = start_ts + INTERVAL_SEC

        hist_yes = fetch_price_history(token_yes, start_ts, end_ts)
        hist_no  = fetch_price_history(token_no,  start_ts, end_ts)

        if hist_yes is None or hist_no is None:
            raise ValueError("price_history_fetch_failed")

        dict_yes = minute_elapsed_dict(start_ts, hist_yes)
        dict_no  = minute_elapsed_dict(start_ts, hist_no)
        mid = {
            k: (dict_yes[k] + (1 - dict_no[k])) / 2
            for k in dict_yes.keys() & dict_no.keys()
        }

        outcome = int(float(outcome_prices[yes_idx]) > 0.5)

        return True, {
            **row,
            "price_history_yes": dict_yes,
            "price_history_no":  dict_no,
            "price_history_mid": mid,
            "outcome": outcome,
        }

    except Exception as e:
        return False, {**row, "reason": str(e)}


# =========================================================
# SAVE
# =========================================================
def save_merged(df_existing: pd.DataFrame, df_new: pd.DataFrame):
    if df_new.empty:
        print("[Save] 신규 데이터 없음, 저장 스킵")
        return df_existing

    if df_existing.empty:
        df_out = df_new
    else:
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
        df_out = df_out.drop_duplicates(subset=["asset", "slot_epoch"]).reset_index(drop=True)

    df_out = df_out.sort_values(["asset", "slot_epoch"]).reset_index(drop=True)
    df_out.to_pickle(PKL_PATH)

    summary_cols = [c for c in ["asset", "slot_utc", "slug", "outcome", "volume"] if c in df_out.columns]
    df_out[summary_cols].to_csv(SUMMARY_PATH, index=False)

    print(f"[Save] 저장 완료 → {PKL_PATH}  (총 {len(df_out)}행, 신규 {len(df_new)}행)")
    return df_out


# =========================================================
# MAIN
# =========================================================
def main():
    start_dt    = parse_utc(START_DATETIME_UTC)
    start_epoch = floor_to_5m_epoch(dt_to_epoch(start_dt))

    if END_DATETIME_UTC is None:
        now_utc   = datetime.now(timezone.utc)
        end_epoch = floor_to_5m_epoch(dt_to_epoch(now_utc)) - INTERVAL_SEC  # 진행 중 봉 제외
    else:
        end_epoch = floor_to_5m_epoch(dt_to_epoch(parse_utc(END_DATETIME_UTC))) - INTERVAL_SEC

    slots = list(range(start_epoch, end_epoch + INTERVAL_SEC, INTERVAL_SEC))

    print("=" * 60)
    print("Polymarket 5m Market Collector (증분 업데이트)")
    print("=" * 60)
    print(f"Start UTC : {datetime.fromtimestamp(start_epoch, tz=timezone.utc)}")
    print(f"End UTC   : {datetime.fromtimestamp(end_epoch,   tz=timezone.utc)}")
    print(f"Slots     : {len(slots)}")
    print(f"Assets    : {list(ASSETS.values())}")
    print(f"Total jobs: {len(slots) * len(ASSETS)}")
    print("=" * 60)

    df_existing, existing_keys = load_existing()

    all_jobs  = [(a_short, a_full, s) for s in slots for a_short, a_full in ASSETS.items()]
    todo_jobs = [(a_short, a_full, s) for a_short, a_full, s in all_jobs
                 if (a_full, s) not in existing_keys]

    print(f"[Filter] 전체 {len(all_jobs)}개 중 기존 {len(existing_keys)}개 스킵 → 신규 {len(todo_jobs)}개 수집")

    if not todo_jobs:
        print("[Done] 수집할 신규 슬롯 없음 ✅")
        return

    # STEP 1: Meta
    meta_ok, meta_miss = [], []
    with tqdm(total=len(todo_jobs), desc="Step1 Meta") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_meta_one_slot, a_s, a_f, s): (a_s, a_f, s)
                    for a_s, a_f, s in todo_jobs}
            for fut in as_completed(futs):
                ok, res = fut.result()
                (meta_ok if ok else meta_miss).append(res)
                pbar.update(1)

    print(f"[Step1] 성공 {len(meta_ok)} / 실패 {len(meta_miss)}")

    if not meta_ok:
        print("[Done] Meta 수집 결과 없음")
        return

    df_meta = pd.DataFrame(meta_ok)

    # STEP 2: Price History
    price_ok, price_miss = [], []
    with tqdm(total=len(df_meta), desc="Step2 Price") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_price_history_one_row, r): r for r in df_meta.to_dict("records")}
            for fut in as_completed(futs):
                ok, res = fut.result()
                (price_ok if ok else price_miss).append(res)
                pbar.update(1)

    print(f"[Step2] 성공 {len(price_ok)} / 실패 {len(price_miss)}")

    if price_miss:
        miss_rows = meta_miss + price_miss
        if MISSING_PATH.exists():
            try:
                df_prev_miss = pd.read_pickle(MISSING_PATH)
                miss_rows = pd.concat([df_prev_miss, pd.DataFrame(miss_rows)], ignore_index=True).to_dict("records")
            except Exception:
                pass
        pd.DataFrame(miss_rows).to_pickle(MISSING_PATH)

    df_new = pd.DataFrame(price_ok) if price_ok else pd.DataFrame()
    save_merged(df_existing, df_new)

    print("\n[DONE] 완료")


if __name__ == "__main__":
    main()

Polymarket 5m Market Collector (증분 업데이트)
Start UTC : 2026-02-19 00:00:00+00:00
End UTC   : 2026-04-30 23:55:00+00:00
Slots     : 20448
Assets    : ['bitcoin', 'ethereum', 'solana', 'xrp']
Total jobs: 81792
[Filter] 전체 81792개 중 기존 0개 스킵 → 신규 81792개 수집


Step1 Meta:   0%|          | 0/81792 [00:00<?, ?it/s]

[Step1] 성공 81732 / 실패 60


Step2 Price:   0%|          | 0/81732 [00:00<?, ?it/s]

[Step2] 성공 81731 / 실패 1
[Save] 저장 완료 → polymarket_meta_5m\5m_closed_only.pkl  (총 81731행, 신규 81731행)

[DONE] 완료


In [2]:
from pathlib import Path
import pandas as pd
import ast

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
BASE_DIR = (POLY_DATA_DIR / "polymarket_meta_5m").resolve()
df_ok = pd.read_pickle(BASE_DIR / "5m_closed_only.pkl")
df_ok = df_ok.sort_values('slot_epoch').drop('price_history_mid', axis=1, errors='ignore').iloc[8:]

def parse_dict(x):
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return {}

def all_half(x):
    d = parse_dict(x)
    if not d:
        return False
    return all(v == 0.5 for v in d.values())

most_common_len = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))).mode()[0]
mask_abnormal = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))) != most_common_len
mask_all05    = df_ok['price_history_yes'].apply(all_half)

df_clean = df_ok[~mask_abnormal & ~mask_all05].reset_index(drop=True)

print(f"원본: {len(df_ok):,}개")
print(f"제거 (길이 다름): {mask_abnormal.sum():,}개")
print(f"제거 (전부 0.5):  {mask_all05.sum():,}개")
print(f"최종: {len(df_clean):,}개")

df_clean.to_pickle(BASE_DIR / "5m_closed_only_clean.pkl")
print("저장 완료 → 5m_closed_only_clean.pkl")

원본: 81,723개
제거 (길이 다름): 7,123개
제거 (전부 0.5):  29개
최종: 74,575개
저장 완료 → 5m_closed_only_clean.pkl


# 15m

In [3]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from tqdm.auto import tqdm


# =========================================================
# CONFIG
# =========================================================
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
OUT_DIR = POLY_DATA_DIR / "polymarket_meta_monthly"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = ["btc", "eth", "sol", "xrp"]

START_DATETIME_UTC = "2025-06-11 00:00:00"   # ← 원하는 시작일로 변경
END_DATETIME_UTC   = "2026-05-01 00:00:00"             

GAMMA_MARKET_BY_SLUG = "https://gamma-api.polymarket.com/markets/slug"
CLOB_PRICE_HISTORY   = "https://clob.polymarket.com/prices-history"

TIMEOUT             = 12
MAX_WORKERS         = 16
SLEEP_BETWEEN_CALLS = 0.00
FIDELITY            = 1

PKL_PATH     = OUT_DIR / "15m_closed_only.pkl"
MISSING_PATH = OUT_DIR / "15m_missing.pkl"
SUMMARY_PATH = OUT_DIR / "15m_closed_only_summary.csv"


# =========================================================
# TIME UTILS
# =========================================================
def parse_utc(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)

def dt_to_epoch(dt: datetime) -> int:
    return int(dt.timestamp())

def floor_to_15m_epoch(ts: int) -> int:
    return ts - (ts % 900)

def epoch_to_slot_utc(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()

def generate_slug(asset: str, slot_epoch: int) -> str:
    # 15분봉: {asset}-updown-15m-{slot_epoch}
    return f"{asset}-updown-15m-{slot_epoch}"

def iso_to_epoch(ts: str) -> int:
    return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp())

def minute_left_dict(end_ts: int, history: list, window_minutes: int = 15):
    d = {}
    for h in history:
        m_left = int((end_ts - h["t"]) // 60)
        if 0 <= m_left <= window_minutes:
            d[m_left] = h["p"]
    return d


# =========================================================
# 기존 데이터 로드
# =========================================================
def load_existing() -> tuple[pd.DataFrame, set]:
    if PKL_PATH.exists():
        try:
            df = pd.read_pickle(PKL_PATH)
            existing = set(zip(df["asset"], df["slot_epoch"]))
            print(f"[Load] 기존 데이터 {len(df)}행 로드 완료 ({len(existing)}개 (asset, slot) 쌍)")
            return df, existing
        except Exception as e:
            print(f"[Load] pkl 로드 실패: {e} → 빈 상태로 시작")
    return pd.DataFrame(), set()


# =========================================================
# HTTP
# =========================================================
def fetch_market_meta(slug: str):
    try:
        r = requests.get(f"{GAMMA_MARKET_BY_SLUG}/{slug}", timeout=TIMEOUT)
        if r.status_code != 200:
            return None, r.status_code
        return r.json(), 200
    except Exception:
        return None, -1

def fetch_price_history(token_id: str, start_ts: int, end_ts: int):
    params = {"market": token_id, "startTs": start_ts, "endTs": end_ts, "fidelity": FIDELITY}
    try:
        r = requests.get(CLOB_PRICE_HISTORY, params=params, timeout=TIMEOUT)
        if r.status_code != 200:
            return None
        return r.json().get("history", [])
    except Exception:
        return None


# =========================================================
# STEP 1: Meta 수집
# =========================================================
def collect_meta_one_slot(asset: str, slot_epoch: int):
    slug     = generate_slug(asset, slot_epoch)
    slot_utc = epoch_to_slot_utc(slot_epoch)

    if SLEEP_BETWEEN_CALLS:
        time.sleep(SLEEP_BETWEEN_CALLS)

    meta, status = fetch_market_meta(slug)

    if meta is None:
        return False, {
            "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": status, "reason": "http_failed",
        }

    clobs = meta.get("clobTokenIds")
    if clobs is None:
        return False, {
            "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": 200, "reason": "no_clobTokenIds",
        }

    return True, {
        "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
        "slug": slug, "clobTokenIds": clobs,
        "outcomes":      meta.get("outcomes"),
        "outcomePrices": meta.get("outcomePrices"),
        "volume": float(meta["volume"]) if meta.get("volume") is not None else None,
    }


# =========================================================
# STEP 2: Price History 수집
# =========================================================
def collect_price_history_one_row(row: dict):
    try:
        clobs          = json.loads(row["clobTokenIds"]) if isinstance(row["clobTokenIds"], str) else row["clobTokenIds"]
        outcomes       = json.loads(row["outcomes"])     if isinstance(row["outcomes"], str)     else row["outcomes"]
        outcome_prices = json.loads(row["outcomePrices"])if isinstance(row["outcomePrices"], str)else row["outcomePrices"]

        yes_idx   = outcomes.index("Up") if "Up" in outcomes else 0
        no_idx    = 1 - yes_idx
        token_yes = clobs[yes_idx]
        token_no  = clobs[no_idx]

        end_ts   = iso_to_epoch(row["slot_utc"]) + 900   # 15분 후
        start_ts = end_ts - 900

        hist_yes = fetch_price_history(token_yes, start_ts, end_ts)
        hist_no  = fetch_price_history(token_no,  start_ts, end_ts)

        if hist_yes is None or hist_no is None:
            raise ValueError("price_history_fetch_failed")

        dict_yes = minute_left_dict(end_ts, hist_yes)
        dict_no  = minute_left_dict(end_ts, hist_no)
        mid = {
            k: (dict_yes[k] + (1 - dict_no[k])) / 2
            for k in dict_yes.keys() & dict_no.keys()
        }

        outcome = int(float(outcome_prices[yes_idx]) > 0.5)

        return True, {
            **row,
            "price_history_yes": dict_yes,
            "price_history_no":  dict_no,
            "price_history_mid": mid,
            "outcome": outcome,
        }

    except Exception as e:
        return False, {**row, "reason": str(e)}


# =========================================================
# SAVE
# =========================================================
def save_merged(df_existing: pd.DataFrame, df_new: pd.DataFrame):
    if df_new.empty:
        print("[Save] 신규 데이터 없음, 저장 스킵")
        return df_existing

    if df_existing.empty:
        df_out = df_new
    else:
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
        df_out = df_out.drop_duplicates(subset=["asset", "slot_epoch"]).reset_index(drop=True)

    df_out = df_out.sort_values(["asset", "slot_epoch"]).reset_index(drop=True)
    df_out.to_pickle(PKL_PATH)

    summary_cols = [c for c in ["asset", "slot_utc", "slug", "outcome", "volume"] if c in df_out.columns]
    df_out[summary_cols].to_csv(SUMMARY_PATH, index=False)

    print(f"[Save] 저장 완료 → {PKL_PATH}  (총 {len(df_out)}행, 신규 {len(df_new)}행)")
    return df_out


# =========================================================
# MAIN
# =========================================================
def main():
    start_dt    = parse_utc(START_DATETIME_UTC)
    start_epoch = floor_to_15m_epoch(dt_to_epoch(start_dt))

    if END_DATETIME_UTC is None:
        now_utc   = datetime.now(timezone.utc)
        end_epoch = floor_to_15m_epoch(dt_to_epoch(now_utc)) - 900  # 현재 진행 중인 봉 제외
    else:
        end_epoch = floor_to_15m_epoch(dt_to_epoch(parse_utc(END_DATETIME_UTC))) - 900

    slots = list(range(start_epoch, end_epoch + 900, 900))

    print("=" * 60)
    print("Polymarket 15m Market Collector (증분 업데이트)")
    print("=" * 60)
    print(f"Start UTC : {datetime.fromtimestamp(start_epoch, tz=timezone.utc)}")
    print(f"End UTC   : {datetime.fromtimestamp(end_epoch,   tz=timezone.utc)}")
    print(f"Slots     : {len(slots)}")
    print(f"Assets    : {ASSETS}")
    print(f"Total jobs: {len(slots) * len(ASSETS)}")
    print("=" * 60)

    df_existing, existing_keys = load_existing()

    all_jobs  = [(a, s) for s in slots for a in ASSETS]
    todo_jobs = [(a, s) for a, s in all_jobs if (a, s) not in existing_keys]

    print(f"[Filter] 전체 {len(all_jobs)}개 중 기존 {len(existing_keys)}개 스킵 → 신규 {len(todo_jobs)}개 수집")

    if not todo_jobs:
        print("[Done] 수집할 신규 슬롯 없음 ✅")
        return

    # STEP 1: Meta
    meta_ok, meta_miss = [], []
    with tqdm(total=len(todo_jobs), desc="Step1 Meta") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_meta_one_slot, a, s): (a, s) for a, s in todo_jobs}
            for fut in as_completed(futs):
                ok, res = fut.result()
                if ok:
                    meta_ok.append(res)
                else:
                    meta_miss.append(res)
                pbar.update(1)

    print(f"[Step1] 성공 {len(meta_ok)} / 실패 {len(meta_miss)}")

    if not meta_ok:
        print("[Done] Meta 수집 결과 없음")
        return

    df_meta = pd.DataFrame(meta_ok)

    # STEP 2: Price History
    price_ok, price_miss = [], []
    with tqdm(total=len(df_meta), desc="Step2 Price") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_price_history_one_row, r): r for r in df_meta.to_dict("records")}
            for fut in as_completed(futs):
                ok, res = fut.result()
                (price_ok if ok else price_miss).append(res)
                pbar.update(1)

    print(f"[Step2] 성공 {len(price_ok)} / 실패 {len(price_miss)}")

    if price_miss:
        miss_rows = meta_miss + price_miss
        if MISSING_PATH.exists():
            try:
                df_prev_miss = pd.read_pickle(MISSING_PATH)
                miss_rows = pd.concat([df_prev_miss, pd.DataFrame(miss_rows)], ignore_index=True).to_dict("records")
            except Exception:
                pass
        pd.DataFrame(miss_rows).to_pickle(MISSING_PATH)

    df_new = pd.DataFrame(price_ok) if price_ok else pd.DataFrame()
    save_merged(df_existing, df_new)

    print("\n[DONE] 완료")


if __name__ == "__main__":
    main()

Polymarket 15m Market Collector (증분 업데이트)
Start UTC : 2025-06-11 00:00:00+00:00
End UTC   : 2026-04-30 23:45:00+00:00
Slots     : 31104
Assets    : ['btc', 'eth', 'sol', 'xrp']
Total jobs: 124416
[Filter] 전체 124416개 중 기존 0개 스킵 → 신규 124416개 수집


Step1 Meta:   0%|          | 0/124416 [00:00<?, ?it/s]

[Step1] 성공 74504 / 실패 49912


Step2 Price:   0%|          | 0/74504 [00:00<?, ?it/s]

[Step2] 성공 73766 / 실패 738
[Save] 저장 완료 → polymarket_meta_monthly\15m_closed_only.pkl  (총 73766행, 신규 73766행)

[DONE] 완료


In [4]:
from pathlib import Path
import pandas as pd
import ast

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
BASE_DIR = (POLY_DATA_DIR / "polymarket_meta_monthly").resolve()
df_ok = pd.read_pickle(BASE_DIR / "15m_closed_only.pkl")
df_ok = df_ok.sort_values('slot_epoch').drop('price_history_mid', axis=1, errors='ignore')

def parse_dict(x):
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return {}

def all_half(x):
    d = parse_dict(x)
    if not d:
        return False
    return all(v == 0.5 for v in d.values())

most_common_len = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))).mode()[0]
mask_abnormal = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))) != most_common_len
mask_all05    = df_ok['price_history_yes'].apply(all_half)

df_clean = df_ok[~mask_abnormal & ~mask_all05].reset_index(drop=True)

print(f"원본: {len(df_ok):,}개")
print(f"제거 (길이 다름): {mask_abnormal.sum():,}개")
print(f"제거 (전부 0.5):  {mask_all05.sum():,}개")
print(f"최종: {len(df_clean):,}개")

df_clean.to_pickle(BASE_DIR / "15m_closed_only_clean.pkl")
print("저장 완료 → 15m_closed_only_clean.pkl")

원본: 73,766개
제거 (길이 다름): 5,335개
제거 (전부 0.5):  60개
최종: 68,376개
저장 완료 → 15m_closed_only_clean.pkl


# 1h

In [5]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from zoneinfo import ZoneInfo

import requests
import pandas as pd
from tqdm.auto import tqdm


# =========================================================
# CONFIG
# =========================================================
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
OUT_DIR = POLY_DATA_DIR / "polymarket_meta_hourly"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = ["bitcoin", "ethereum", "solana", "xrp"]

START_DATETIME_UTC = "2025-02-19 00:00:00"   # <- 원하는 시작일로 변경
END_DATETIME_UTC   = "2026-05-01 00:00:00"                     # <- None이면 현재 시각 기준, "2026-01-01 00:00:00" 형식으로 지정 가능

GAMMA_MARKET_BY_SLUG = "https://gamma-api.polymarket.com/markets/slug"
CLOB_PRICE_HISTORY   = "https://clob.polymarket.com/prices-history"

TIMEOUT             = 12
MAX_WORKERS         = 8
SLEEP_BETWEEN_CALLS = 0.05
FIDELITY            = 1

ET = ZoneInfo("America/New_York")

# slug 포맷 변경 분기점
SLUG_YEAR_EPOCH    = 1773550800
SLUG_NOYEAR_EPOCH  = 1774472400
SLUG_YEAR2_EPOCH   = 1774479600
SLUG_NOYEAR2_EPOCH = 1775188800
SLUG_YEAR3_EPOCH   = 1775239200

# 마켓이 열리지 않은 슬롯
MISSING_SLOTS = {
    1774472400,
    1775188800,
    1775192400,
    1775196000,
    1775199600,
    1775203200,
    1775206800,
    1775210400,
    1775232000,
    1775235600,
}

PKL_PATH     = OUT_DIR / "hourly_closed_only.pkl"
MISSING_PATH = OUT_DIR / "hourly_missing.pkl"
SUMMARY_PATH = OUT_DIR / "hourly_closed_only_summary.csv"


# =========================================================
# TIME UTILS
# =========================================================
def parse_utc(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)

def dt_to_epoch(dt: datetime) -> int:
    return int(dt.timestamp())

def floor_to_1h_epoch(ts: int) -> int:
    return ts - (ts % 3600)

def epoch_to_slot_utc(ts: int) -> str:
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()

def generate_slug(asset: str, dt_utc: datetime, slot_epoch: int) -> str:
    dt_et      = dt_utc.astimezone(ET)
    month_name = dt_et.strftime("%B").lower()
    day        = dt_et.day
    hour       = dt_et.hour
    if hour == 0:      hour_str = "12am"
    elif hour < 12:    hour_str = f"{hour}am"
    elif hour == 12:   hour_str = "12pm"
    else:              hour_str = f"{hour - 12}pm"

    use_year = (
        (SLUG_YEAR_EPOCH  <= slot_epoch < SLUG_NOYEAR_EPOCH)  or
        (SLUG_YEAR2_EPOCH <= slot_epoch < SLUG_NOYEAR2_EPOCH) or
        (slot_epoch >= SLUG_YEAR3_EPOCH)
    )

    if use_year:
        return f"{asset}-up-or-down-{month_name}-{day}-{dt_et.year}-{hour_str}-et"
    else:
        return f"{asset}-up-or-down-{month_name}-{day}-{hour_str}-et"

def iso_to_epoch(ts: str) -> int:
    return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp())

def minute_left_dict(end_ts: int, history: list, window_minutes: int = 60):
    d = {}
    for h in history:
        m_left = int((end_ts - h["t"]) // 60)
        if 0 <= m_left <= window_minutes:
            d[m_left] = h["p"]
    return d


# =========================================================
# 기존 데이터 로드
# =========================================================
def load_existing() -> tuple[pd.DataFrame, set]:
    if PKL_PATH.exists():
        try:
            df = pd.read_pickle(PKL_PATH)
            existing = set(zip(df["asset"], df["slot_epoch"]))
            print(f"[Load] 기존 데이터 {len(df)}행 로드 완료 ({len(existing)}개 (asset, slot) 쌍)")
            return df, existing
        except Exception as e:
            print(f"[Load] pkl 로드 실패: {e} -> 빈 상태로 시작")
    return pd.DataFrame(), set()


# =========================================================
# HTTP
# =========================================================
def fetch_market_meta(slug: str):
    try:
        r = requests.get(f"{GAMMA_MARKET_BY_SLUG}/{slug}", timeout=TIMEOUT)
        if r.status_code != 200:
            return None, r.status_code
        return r.json(), 200
    except Exception:
        return None, -1

def fetch_price_history(token_id: str, start_ts: int, end_ts: int):
    params = {"market": token_id, "startTs": start_ts, "endTs": end_ts, "fidelity": FIDELITY}
    try:
        r = requests.get(CLOB_PRICE_HISTORY, params=params, timeout=TIMEOUT)
        if r.status_code != 200:
            return None
        return r.json().get("history", [])
    except Exception:
        return None


# =========================================================
# STEP 1: Meta 수집
# =========================================================
def collect_meta_one_slot(asset: str, slot_epoch: int):
    if slot_epoch in MISSING_SLOTS:
        return False, None

    dt_utc   = datetime.fromtimestamp(slot_epoch, tz=timezone.utc)
    slug     = generate_slug(asset, dt_utc, slot_epoch)
    slot_utc = epoch_to_slot_utc(slot_epoch)

    if SLEEP_BETWEEN_CALLS:
        time.sleep(SLEEP_BETWEEN_CALLS)

    meta, status = fetch_market_meta(slug)

    if meta is None:
        return False, {
            "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": status, "reason": "http_failed",
        }

    clobs = meta.get("clobTokenIds")
    if clobs is None:
        return False, {
            "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
            "slug": slug, "status": 200, "reason": "no_clobTokenIds",
        }

    return True, {
        "asset": asset, "slot_epoch": slot_epoch, "slot_utc": slot_utc,
        "slug": slug, "clobTokenIds": clobs,
        "outcomes":      meta.get("outcomes"),
        "outcomePrices": meta.get("outcomePrices"),
        "volume": float(meta["volume"]) if meta.get("volume") is not None else None,
    }


# =========================================================
# STEP 2: Price History 수집
# =========================================================
def collect_price_history_one_row(row: dict):
    try:
        clobs          = json.loads(row["clobTokenIds"]) if isinstance(row["clobTokenIds"], str) else row["clobTokenIds"]
        outcomes       = json.loads(row["outcomes"])     if isinstance(row["outcomes"], str)     else row["outcomes"]
        outcome_prices = json.loads(row["outcomePrices"])if isinstance(row["outcomePrices"], str)else row["outcomePrices"]

        yes_idx   = outcomes.index("Up") if "Up" in outcomes else 0
        no_idx    = 1 - yes_idx
        token_yes = clobs[yes_idx]
        token_no  = clobs[no_idx]

        end_ts   = iso_to_epoch(row["slot_utc"]) + 3600
        start_ts = end_ts - 3600

        hist_yes = fetch_price_history(token_yes, start_ts, end_ts)
        hist_no  = fetch_price_history(token_no,  start_ts, end_ts)

        if hist_yes is None or hist_no is None:
            raise ValueError("price_history_fetch_failed")

        dict_yes = minute_left_dict(end_ts, hist_yes)
        dict_no  = minute_left_dict(end_ts, hist_no)
        mid = {
            k: (dict_yes[k] + (1 - dict_no[k])) / 2
            for k in dict_yes.keys() & dict_no.keys()
        }

        outcome = int(float(outcome_prices[yes_idx]) > 0.5)

        return True, {
            **row,
            "price_history_yes": dict_yes,
            "price_history_no":  dict_no,
            "price_history_mid": mid,
            "outcome": outcome,
        }

    except Exception as e:
        return False, {**row, "reason": str(e)}


# =========================================================
# SAVE
# =========================================================
def save_merged(df_existing: pd.DataFrame, df_new: pd.DataFrame):
    if df_new.empty:
        print("[Save] 신규 데이터 없음, 저장 스킵")
        return df_existing

    if df_existing.empty:
        df_out = df_new
    else:
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
        df_out = df_out.drop_duplicates(subset=["asset", "slot_epoch"]).reset_index(drop=True)

    df_out = df_out.sort_values(["asset", "slot_epoch"]).reset_index(drop=True)
    df_out.to_pickle(PKL_PATH)

    summary_cols = [c for c in ["asset", "slot_utc", "slug", "outcome", "volume"] if c in df_out.columns]
    df_out[summary_cols].to_csv(SUMMARY_PATH, index=False)

    print(f"[Save] 저장 완료 -> {PKL_PATH}  (총 {len(df_out)}행, 신규 {len(df_new)}행)")
    return df_out


# =========================================================
# MAIN
# =========================================================
def main():
    start_dt    = parse_utc(START_DATETIME_UTC)
    start_epoch = floor_to_1h_epoch(dt_to_epoch(start_dt))

    if END_DATETIME_UTC is None:
        now_utc   = datetime.now(timezone.utc)
        end_epoch = floor_to_1h_epoch(dt_to_epoch(now_utc)) - 3600
    else:
        end_epoch = floor_to_1h_epoch(dt_to_epoch(parse_utc(END_DATETIME_UTC))) - 3600

    slots = list(range(start_epoch, end_epoch + 3600, 3600))

    print("=" * 60)
    print("Polymarket 1h Market Collector (DST 대응 + 증분 업데이트)")
    print("=" * 60)
    print(f"Start UTC : {datetime.fromtimestamp(start_epoch, tz=timezone.utc)}")
    print(f"End UTC   : {datetime.fromtimestamp(end_epoch,   tz=timezone.utc)}")
    print(f"Slots     : {len(slots)}")
    print(f"Assets    : {ASSETS}")
    print(f"Total jobs: {len(slots) * len(ASSETS)}")
    print(f"Missing slots (skip): {sorted(MISSING_SLOTS)}")
    print("=" * 60)

    df_existing, existing_keys = load_existing()

    all_jobs  = [(a, s) for s in slots for a in ASSETS]
    todo_jobs = [(a, s) for a, s in all_jobs
                 if (a, s) not in existing_keys and s not in MISSING_SLOTS]

    print(f"[Filter] 전체 {len(all_jobs)}개 중 기존 {len(existing_keys)}개 + "
          f"missing {len(MISSING_SLOTS)}개 스킵 -> 신규 {len(todo_jobs)}개 수집")

    if not todo_jobs:
        print("[Done] 수집할 신규 슬롯 없음 ✅")
        return

    # STEP 1: Meta
    meta_ok, meta_miss = [], []
    with tqdm(total=len(todo_jobs), desc="Step1 Meta") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_meta_one_slot, a, s): (a, s) for a, s in todo_jobs}
            for fut in as_completed(futs):
                ok, res = fut.result()
                if res is None:
                    pass
                elif ok:
                    meta_ok.append(res)
                else:
                    meta_miss.append(res)
                pbar.update(1)

    print(f"[Step1] 성공 {len(meta_ok)} / 실패 {len(meta_miss)}")

    if not meta_ok:
        print("[Done] Meta 수집 결과 없음")
        return

    df_meta = pd.DataFrame(meta_ok)

    # STEP 2: Price History
    price_ok, price_miss = [], []
    with tqdm(total=len(df_meta), desc="Step2 Price") as pbar:
        with ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(collect_price_history_one_row, r): r for r in df_meta.to_dict("records")}
            for fut in as_completed(futs):
                ok, res = fut.result()
                (price_ok if ok else price_miss).append(res)
                pbar.update(1)

    print(f"[Step2] 성공 {len(price_ok)} / 실패 {len(price_miss)}")

    if price_miss:
        miss_rows = meta_miss + price_miss
        if MISSING_PATH.exists():
            try:
                df_prev_miss = pd.read_pickle(MISSING_PATH)
                miss_rows = pd.concat([df_prev_miss, pd.DataFrame(miss_rows)], ignore_index=True).to_dict("records")
            except Exception:
                pass
        pd.DataFrame(miss_rows).to_pickle(MISSING_PATH)

    df_new = pd.DataFrame(price_ok) if price_ok else pd.DataFrame()
    save_merged(df_existing, df_new)

    print("\n[DONE] 완료")


if __name__ == "__main__":
    main()

Polymarket 1h Market Collector (DST 대응 + 증분 업데이트)
Start UTC : 2025-02-19 00:00:00+00:00
End UTC   : 2026-04-30 23:00:00+00:00
Slots     : 10464
Assets    : ['bitcoin', 'ethereum', 'solana', 'xrp']
Total jobs: 41856
Missing slots (skip): [1774472400, 1775188800, 1775192400, 1775196000, 1775199600, 1775203200, 1775206800, 1775210400, 1775232000, 1775235600]
[Load] 기존 데이터 10622행 로드 완료 (10622개 (asset, slot) 쌍)
[Filter] 전체 41856개 중 기존 10622개 + missing 10개 스킵 -> 신규 31194개 수집


Step1 Meta:   0%|          | 0/31194 [00:00<?, ?it/s]

[Step1] 성공 21928 / 실패 9266


Step2 Price:   0%|          | 0/21928 [00:00<?, ?it/s]

[Step2] 성공 21910 / 실패 18
[Save] 저장 완료 -> polymarket_meta_hourly\hourly_closed_only.pkl  (총 32532행, 신규 21910행)

[DONE] 완료


In [6]:
from pathlib import Path
import pandas as pd
import ast

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"
BASE_DIR = (POLY_DATA_DIR / "polymarket_meta_hourly").resolve()
df_ok = pd.read_pickle(BASE_DIR / "hourly_closed_only.pkl")
df_ok = df_ok.sort_values('slot_epoch').drop('price_history_mid', axis=1, errors='ignore')

def parse_dict(x):
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return {}

def all_half(x):
    d = parse_dict(x)
    if not d:
        return False
    return all(v == 0.5 for v in d.values())

most_common_len = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))).mode()[0]
mask_abnormal = df_ok['price_history_yes'].apply(lambda x: len(parse_dict(x))) != most_common_len
mask_all05    = df_ok['price_history_yes'].apply(all_half)

df_clean = df_ok[~mask_abnormal & ~mask_all05].reset_index(drop=True)

print(f"원본: {len(df_ok):,}개")
print(f"제거 (길이 다름): {mask_abnormal.sum():,}개")
print(f"제거 (전부 0.5):  {mask_all05.sum():,}개")
print(f"최종: {len(df_clean):,}개")

df_clean.to_pickle(BASE_DIR / "hourly_closed_only_clean.pkl")
print("저장 완료 → hourly_closed_only_clean.pkl")

원본: 32,532개
제거 (길이 다름): 5,174개
제거 (전부 0.5):  3개
최종: 27,355개
저장 완료 → hourly_closed_only_clean.pkl


# 결과

In [11]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "poly data").exists()), Path.cwd())
POLY_DATA_DIR = PROJECT_ROOT / "poly data"

configs = [
    ("5분봉",  (POLY_DATA_DIR / "polymarket_meta_5m").resolve(),      "5m_closed_only_clean.pkl",      "bitcoin", "ethereum", "solana", "xrp"),
    ("15분봉", (POLY_DATA_DIR / "polymarket_meta_monthly").resolve(),  "15m_closed_only_clean.pkl",     "btc",     "eth",      "sol",    "xrp"),
    ("1시간봉", (POLY_DATA_DIR / "polymarket_meta_hourly").resolve(),  "hourly_closed_only_clean.pkl",  "bitcoin", "ethereum", "solana", "xrp"),
]

for tf, base_dir, fname, btc, eth, sol, xrp in configs:
    print(f"\n{'='*50}")
    print(f"[{tf}]")
    df = pd.read_pickle(base_dir / fname)
    for name, asset in [('BTC', btc), ('ETH', eth), ('SOL', sol), ('XRP', xrp)]:
        d = df[df['asset'] == asset]
        print(f"  {name}: {len(d):,}개 | {d['slot_utc'].min()} ~ {d['slot_utc'].max()}")


[5분봉]
  BTC: 18,621개 | 2026-02-19T00:10:00+00:00 ~ 2026-04-30T23:55:00+00:00
  ETH: 18,666개 | 2026-02-19T00:10:00+00:00 ~ 2026-04-30T23:55:00+00:00
  SOL: 18,647개 | 2026-02-19T00:10:00+00:00 ~ 2026-04-30T23:55:00+00:00
  XRP: 18,641개 | 2026-02-19T00:10:00+00:00 ~ 2026-04-30T23:55:00+00:00

[15분봉]
  BTC: 17,985개 | 2025-10-09T16:45:00+00:00 ~ 2026-04-30T23:45:00+00:00
  ETH: 17,957개 | 2025-10-09T16:45:00+00:00 ~ 2026-04-30T23:45:00+00:00
  SOL: 16,218개 | 2025-10-27T20:00:00+00:00 ~ 2026-04-30T23:45:00+00:00
  XRP: 16,216개 | 2025-10-27T19:45:00+00:00 ~ 2026-04-30T23:45:00+00:00

[1시간봉]
  BTC: 6,897개 | 2025-06-11T04:00:00+00:00 ~ 2026-04-30T23:00:00+00:00
  ETH: 6,867개 | 2025-06-11T04:00:00+00:00 ~ 2026-04-30T23:00:00+00:00
  SOL: 6,864개 | 2025-06-17T04:00:00+00:00 ~ 2026-04-30T23:00:00+00:00
  XRP: 6,727개 | 2025-06-24T05:00:00+00:00 ~ 2026-04-30T23:00:00+00:00
